#Import Libraries

In [35]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from google.colab import files

In [36]:
from google.colab import files

uploaded = files.upload()

Saving deep_learning.zip to deep_learning (2).zip


In [37]:
DATASET_DIR = Path("/content/dataset/deep learning")

Train_dir = str(DATASET_DIR / "TRAIN")
Test_dir = str(DATASET_DIR / "TEST")
Val_dir = str(DATASET_DIR / "VAL")

print("Train:", Path(Train_dir).exists())
print("Test:", Path(Test_dir).exists())
print("Validation:", Path(Val_dir).exists())

Train: True
Test: True
Validation: True


#Image Preprocessing

In [38]:


IMG_SIZE = (128, 128)
BATCH_SIZE = 16

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [39]:
train_generator = train_datagen.flow_from_directory(
    Train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    Val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    Test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
NUM_CLASSES = len(class_names)

print("Classes:", class_names)
print("Class mapping:", train_generator.class_indices)
print("Number of classes:", NUM_CLASSES)

Found 27 images belonging to 3 classes.
Found 6 images belonging to 3 classes.
Found 9 images belonging to 3 classes.
Classes: ['AJITH', 'SURIYA', 'VIJAY']
Class mapping: {'AJITH': 0, 'SURIYA': 1, 'VIJAY': 2}
Number of classes: 3


#Model Building

In [40]:
model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.Conv2D(32, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,288,067 (16.36 MB)

 Trainable params: 4,288,067 (16.36 MB)

 Non-trainable params: 0 (0.00 B)

# Model evaluation

In [42]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20
)

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.3333 - loss: 1.4518 - val_accuracy: 0.3333 - val_loss: 1.6621
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - accuracy: 0.2222 - loss: 1.8098 - val_accuracy: 0.0000e+00 - val_loss: 1.1501
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.3704 - loss: 1.0937 - val_accuracy: 0.0000e+00 - val_loss: 1.1391
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step - accuracy: 0.5556 - loss: 1.0343 - val_accuracy: 0.1667 - val_loss: 1.1558
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.5185 - loss: 1.0375 - val_accuracy: 0.1667 - val_loss: 1.2233
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - accuracy: 0.5556 - loss: 0.9972 - val_accuracy: 0.1667 - val_loss: 1.4056
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 0.4815 - loss: 0.9169 - val_accuracy: 0.1667 - val_loss: 1.6533
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - accuracy: 0.6296 - loss: 0.9056 - val_accuracy: 0.3333 - val_

In [47]:
test_loss, test_accuracy = model.evaluate(test_generator)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy * 100, "%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.2222 - loss: 3.3796
Test Loss: 3.3795502185821533
Test Accuracy: 22.22222238779068 %


#  Image Prediction

In [49]:


uploaded = files.upload()

for file in uploaded:
    img = image.load_img(file, target_size=(128, 128))
    img = image.img_to_array(img)
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    result = model.predict(img)

    index = np.argmax(result)
    print("Predicted Actor:", class_names[index])
    print("Accuracy:", round(result[0][index] * 100, 2), "%")

Saving image.jpg to image (1).jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 576ms/step
Predicted Actor: VIJAY
Accuracy: 95.37 %
